In [1]:
# Kiểm tra nhanh các thư viện cơ bản (không bắt buộc tất cả phải có)
import sys, json, math, random, os, time
import numpy as np

try:
    import sklearn
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import classification_report, confusion_matrix
    SKLEARN_OK = True
except Exception as e:
    SKLEARN_OK = False
    print("Thiếu scikit-learn. Các phần dùng sklearn sẽ không chạy:", e)

try:
    import networkx as nx
    NETWORKX_OK = True
except Exception as e:
    NETWORKX_OK = False
    print("Thiếu networkx. Các phần đồ thị sẽ không chạy:", e)

# Các phần tùy chọn (có thể không có internet nên sẽ fail ở đây, cứ để False nếu không)
try:
    import torch
    TORCH_OK = True
except Exception as e:
    TORCH_OK = False

try:
    import transformers
    TRANSFORMERS_OK = True
except Exception as e:
    TRANSFORMERS_OK = False

print("SKLEARN_OK =", SKLEARN_OK, "| NETWORKX_OK =", NETWORKX_OK, "| TORCH_OK =", TORCH_OK, "| TRANSFORMERS_OK =", TRANSFORMERS_OK)


SKLEARN_OK = True | NETWORKX_OK = True | TORCH_OK = True | TRANSFORMERS_OK = True


## 9) ViT (Vision Transformer) – Ứng dụng: **Phát hiện sản phẩm lỗi vs đạt** (demo)

**Mục tiêu thực tế:** Trong dây chuyền sản xuất, ta muốn kiểm tra nhanh ảnh một sản phẩm là *đạt* hay *lỗi*.
Ở đây ta sẽ demo 2 cách:

1. **Tối giản, ngoại tuyến:** Trích xuất **patch embedding kiểu ViT** bằng `numpy` (không train Transformer),
   sau đó huấn luyện **Logistic Regression** để phân loại ảnh *giả lập* (hoa văn/sọc v.v.). Mục tiêu: hiểu pipeline patchify → embedding → phân loại.

2. **Tùy chọn (cần internet):** Dùng ViT pre-trained (`google/vit-base-patch16-224` qua `transformers`) để inference ảnh thật
   (bạn có thể tải hình, hoặc gắn trực tiếp ảnh tại chỗ). Đây là cách thực tế khi có model đã học trước.


In [2]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# ---- Tạo bộ dữ liệu ảnh "giả lập" (64x64 RGB) gồm 2 lớp: 'dat' (0) và 'loi' (1)
# Ý tưởng: lớp 0 có hoa văn 'mịn', lớp 1 có sọc/đốm rõ -> tạo khác biệt cấu trúc cục bộ để patch thấy được.

def make_texture_ok(n=200, size=64, seed=0):
    rng = np.random.default_rng(seed)
    imgs = rng.normal(127, 20, size=(n, size, size, 3)).clip(0,255).astype(np.uint8)
    # Làm mịn nhẹ
    for i in range(n):
        imgs[i] = (imgs[i].astype(np.float32)*0.9 + 12).clip(0,255).astype(np.uint8)
    return imgs

def make_texture_ng(n=200, size=64, seed=1):
    rng = np.random.default_rng(seed)
    imgs = rng.normal(127, 20, size=(n, size, size, 3)).clip(0,255).astype(np.uint8)
    # Thêm sọc mạnh theo trục x
    for i in range(n):
        for x in range(0, size, 4):
            imgs[i, :, x:x+2, :] = 255
    # Thêm đốm nhiễu
    mask = rng.random((n, size, size)) < 0.02
    imgs[mask, :] = 0
    return imgs

X_ok = make_texture_ok(250, 64, seed=42)
X_ng = make_texture_ng(250, 64, seed=43)

X = np.concatenate([X_ok, X_ng], axis=0)
y = np.array([0]*len(X_ok) + [1]*len(X_ng))  # 0: đạt, 1: lỗi

# ---- Patchify: chia ảnh 64x64 thành các patch 8x8 (-> 8x8 = 64 patch). Flatten mỗi patch.
def patchify(img, patch=8):
    H,W,C = img.shape
    assert H%patch==0 and W%patch==0
    patches = []
    for i in range(0, H, patch):
        for j in range(0, W, patch):
            p = img[i:i+patch, j:j+patch, :].reshape(-1)
            patches.append(p)
    return np.stack(patches, axis=0)  # [num_patches, patch*patch*C]

def vit_style_embed(imgs, patch=8, proj_dim=64, seed=0):
    # Mô phỏng ViT embedding: flatten patch -> (linear projection) -> mean pooling toàn bộ patch embedding
    rng = np.random.default_rng(seed)
    P = patch*patch*3
    W = rng.normal(0, 1/np.sqrt(P), size=(P, proj_dim)).astype(np.float32)  # ma trận chiếu ngẫu nhiên cố định
    feats = []
    for img in imgs:
        patches = patchify(img, patch=patch)  # [num_patches, P]
        emb = patches @ W                      # [num_patches, proj_dim]
        cls_like = emb.mean(axis=0)            # giống token [CLS] = trung bình
        feats.append(cls_like)
    return np.stack(feats, axis=0)             # [N, proj_dim]

X_feat = vit_style_embed(X, patch=8, proj_dim=128, seed=123)
X_tr, X_te, y_tr, y_te = train_test_split(X_feat, y, test_size=0.3, random_state=0, stratify=y)

clf = LogisticRegression(max_iter=1000)
clf.fit(X_tr, y_tr)
y_pred = clf.predict(X_te)

print(classification_report(y_te, y_pred, target_names=["dat","loi"]))

              precision    recall  f1-score   support

         dat       1.00      1.00      1.00        75
         loi       1.00      1.00      1.00        75

    accuracy                           1.00       150
   macro avg       1.00      1.00      1.00       150
weighted avg       1.00      1.00      1.00       150



In [3]:
# %pip install -U torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu
# %pip install -U transformers timm pillow

In [4]:
USE_REAL_VIT = True  # đổi thành True nếu bạn đã cài thành công

if USE_REAL_VIT:
    import torch
    from PIL import Image
    from transformers import AutoImageProcessor, ViTForImageClassification

    model_name = "google/vit-base-patch16-224"
    processor = AutoImageProcessor.from_pretrained(model_name)
    model = ViTForImageClassification.from_pretrained(model_name)

    path = r"D:/năm 4/PythonSGU/BaiTap+BaiGiangML/BaiTap/img/dog_alaska.jpg"
    # Tải ảnh mẫu (bạn thay đường dẫn ảnh của bạn vào đây)
    img = Image.open(path).convert("RGB")

    # Demo dùng ảnh tổng hợp từ mảng (chỉ để code không lỗi nếu bạn chưa có ảnh)
    import numpy as np
    arr = np.random.randint(0, 255, (224,224,3), dtype=np.uint8)
    img = Image.fromarray(arr)

    inputs = processor(images=img, return_tensors="pt")
    with torch.no_grad():
        logits = model(**inputs).logits
    pred = logits.argmax(-1).item()
    label = model.config.id2label[pred]
    print("Dự đoán:", label)
else:
    print("Để chạy ViT thật: bật USE_REAL_VIT=True sau khi đã cài torch/transformers.")

preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

c:\Users\CaRot\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\CaRot\.cache\huggingface\hub\models--google--vit-base-patch16-224. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config.json: 0.00B [00:00, ?B/s]

Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Dự đoán: rubber eraser, rubber, pencil eraser
